# 04 - Analyse des features macro

Ce notebook analyse les indicateurs macroéconomiques et leur relation avec les ETF.

**Objectifs :**
- Évolution du VIX et périodes de stress
- Yield curve et prédiction de récession
- Corrélations macro vs returns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

In [ ]:
# Charger les données
data_dir = Path('../data/processed')
df = pd.read_csv(sorted(data_dir.glob('dataset_*.csv'))[-1], index_col=0, parse_dates=True)

macro_cols = ['vix', 'credit_spread_hy', 'yield_curve_10y2y', 't3mo', 't10y', 'wti_oil']
macro = df[[c for c in macro_cols if c in df.columns]].copy()
print(f'Colonnes macro: {list(macro.columns)}')

## 1. VIX - Indice de peur

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# VIX
ax = axes[0]
ax.plot(macro.index, macro['vix'], color='purple', linewidth=1)
ax.axhline(20, color='green', linestyle='--', alpha=0.5, label='Normal (<20)')
ax.axhline(30, color='orange', linestyle='--', alpha=0.5, label='Élevé (>30)')
ax.axhline(40, color='red', linestyle='--', alpha=0.5, label='Panique (>40)')
ax.fill_between(macro.index, macro['vix'], 20, where=macro['vix']>20, color='red', alpha=0.3)
ax.set_ylabel('VIX')
ax.set_title('VIX - Indice de volatilité implicite S&P 500')
ax.legend(loc='upper right')
ax.set_ylim(0, 90)

# SPY pour comparaison
ax = axes[1]
ax.plot(df.index, df['SPY'], color='steelblue', linewidth=1)
ax.set_ylabel('SPY ($)')
ax.set_xlabel('Date')

plt.tight_layout()
plt.show()

print(f"VIX moyen: {macro['vix'].mean():.1f}")
print(f"VIX max: {macro['vix'].max():.1f} ({macro['vix'].idxmax().date()})")

## 2. Yield Curve - Prédicteur de récession

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

yc = macro['yield_curve_10y2y']
ax.plot(yc.index, yc, color='darkblue', linewidth=1)
ax.axhline(0, color='red', linestyle='-', linewidth=2, label='Inversion (récession?)')
ax.fill_between(yc.index, yc, 0, where=yc<0, color='red', alpha=0.4, label='Courbe inversée')
ax.fill_between(yc.index, yc, 0, where=yc>=0, color='green', alpha=0.2)

ax.set_title('Spread 10Y-2Y Treasury (Yield Curve)', fontsize=14)
ax.set_ylabel('Spread (%)')
ax.set_xlabel('Date')
ax.legend()

plt.tight_layout()
plt.show()

inversions = (yc < 0).sum()
print(f"Jours d'inversion: {inversions} ({inversions/len(yc)*100:.1f}%)")

## 3. Corrélations VIX vs Returns

In [ ]:
# Corrélation VIX vs returns SPY
spy_ret = df['SPY'].pct_change()
vix_change = macro['vix'].pct_change()

corr = spy_ret.corr(vix_change)
print(f"Corrélation SPY returns vs VIX changes: {corr:.3f}")

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(vix_change*100, spy_ret*100, alpha=0.3, s=10)
ax.axhline(0, color='black', linestyle='--', alpha=0.3)
ax.axvline(0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('VIX change (%)')
ax.set_ylabel('SPY return (%)')
ax.set_title(f'SPY Returns vs VIX Changes (corr={corr:.2f})')
ax.set_xlim(-30, 50)
ax.set_ylim(-15, 15)
plt.show()

## 4. Régimes de marché (preview Phase 3)

On peut identifier visuellement des régimes :
- **Bull** : VIX < 20, yield curve positive, SPY en hausse
- **Bear** : VIX > 30, drawdowns importants
- **Volatile** : VIX fluctuant, incertitude
- **Stable** : VIX bas et stable

Ces régimes seront modélisés avec un HMM en Phase 3.

In [ ]:
# Classification simple des régimes
regimes = pd.DataFrame(index=df.index)
regimes['vix'] = macro['vix']
regimes['spy_ret_20d'] = df['SPY'].pct_change(20)

def classify_regime(row):
    if pd.isna(row['vix']) or pd.isna(row['spy_ret_20d']):
        return 'Unknown'
    if row['vix'] > 30:
        return 'Stress'
    elif row['vix'] < 15 and row['spy_ret_20d'] > 0.02:
        return 'Bull'
    elif row['spy_ret_20d'] < -0.05:
        return 'Bear'
    else:
        return 'Neutral'

regimes['regime'] = regimes.apply(classify_regime, axis=1)
print(regimes['regime'].value_counts())

## 5. Insights clés

1. **VIX** : Anti-corrélé aux returns SPY (-0.7), pic à 80+ en mars 2020 (COVID)

2. **Yield Curve** : Inversée 2022-2023, historiquement prédicteur de récession 12-18 mois après

3. **Régimes identifiables** : Bull (VIX<15), Stress (VIX>30), transitions visibles

4. **Features utiles pour ML** : VIX, yield curve spread, volatilité réalisée, momentum